# Perplexity Contextual Embeddings: pplx-embed-v2-context-9b-preview

This notebook benchmarks the **unquantized 9B parameter contextual embedding model**:
**`perplexity-ai/pplx-embed-v2-context-9b-preview`**

### Architecture & Execution Highlights:
- **Precision**: Full unquantized **FP16** (16.8 GB weights) sharded across **Dual Nvidia T4 GPUs (T4 x 2)** via `accelerate` (`device_map="auto"`).
- **Contextual Attention**: Contiguous section chunks are passed together so each chunk attends to the global surrounding chapter context during the forward pass.
- **Evaluation**: Evaluates both **15 Exact Technical CLI queries** and **15 Conceptual Zero-Keyword queries** against the 1,243 extracted security book chunks.

> **Prerequisites on Kaggle**:
> 1. Set **Settings -> Accelerator -> GPU T4 x2**
> 2. Set **Settings -> Internet -> On**

In [ ]:
# 1. Install dependencies & initialize safe imports
%pip install -q --upgrade 'transformers>=5.19.0' accelerate safetensors rank_bm25

import sys, types, importlib.abc, importlib.machinery
class _AutoMock(types.ModuleType):
    def __getattr__(self, name):
        if name.startswith('__'): raise AttributeError(name)
        val = _AutoMock(f'{self.__name__}.{name}')
        setattr(self, name, val)
        return val
    def __call__(self, *args, **kwargs): return None
class _MockFinder(importlib.abc.MetaPathFinder):
    def find_spec(self, fullname, path, target=None):
        if fullname == 'torchvision' or fullname.startswith('torchvision.'):
            return importlib.machinery.ModuleSpec(fullname, _MockLoader(), is_package=True)
        return None
class _MockLoader(importlib.abc.Loader):
    def create_module(self, spec):
        mod = _AutoMock(spec.name); mod.__path__ = []; return mod
    def exec_module(self, module): pass
sys.meta_path = [f for f in sys.meta_path if not isinstance(f, _MockFinder)]
sys.meta_path.insert(0, _MockFinder())

In [ ]:
# 2. Unpack embedded dataset (1,243 chunks + Exact & Conceptual Queries)
import gzip, base64, json

CORPUS_B64 = """..."""
raw_corpus_bytes = gzip.decompress(base64.b64decode(CORPUS_B64))
corpus = [json.loads(l) for l in raw_corpus_bytes.decode('utf-8').splitlines() if l.strip()]
exact_queries = json.loads("""[
  {
    "id": "q01",
    "query": "meterpreter privesc getsystem named pipe impersonation",
    "book_tag": "Metasploit",
    "topic": "Privilege Escalation in Meterpreter"
  },
  {
    "id": "q02",
    "query": "shikata ga nai polymorphic shellcode encoder bad characters elimination",
    "book_tag": "Metasploit",
    "topic": "Payload Encoding and Evasion"
  },
  {
    "id": "q03",
    "query": "msfvenom generate staged meterpreter reverse tcp payload windows",
    "book_tag": "Metasploit",
    "topic": "Payload Generation"
  },
  {
    "id": "q04",
    "query": "database fingerprinting differences between mysql and mssql comment syntax",
    "book_tag": "SQLi_Attacks",
    "topic": "SQL Dialect Fingerprinting"
  },
  {
    "id": "q05",
    "query": "blind sql injection timing attacks using WAITFOR DELAY and pg_sleep",
    "book_tag": "SQLi_Attacks",
    "topic": "Time-based Blind SQLi"
  },
  {
    "id": "q06",
    "query": "second order sql injection payload stored in database and executed later",
    "book_tag": "SQLi_Attacks",
    "topic": "Second-Order SQL Injection"
  },
  {
    "id": "q07",
    "query": "out of band sql injection data exfiltration via dns lookup utl_http xp_dirtree",
    "book_tag": "SQLi_Attacks",
    "topic": "Out-of-band Data Exfiltration"
  },
  {
    "id": "q08",
    "query": "bypassing web application firewall using sql comment obfuscation inline comments",
    "book_tag": "SQLi_Attacks",
    "topic": "WAF Evasion via Comment Obfuscation"
  },
  {
    "id": "q09",
    "query": "sql injection in order by clause error based exploitation",
    "book_tag": "SQLi_Attacks",
    "topic": "Order By Clause Injection"
  },
  {
    "id": "q10",
    "query": "reflected cross site scripting vs stored xss execution context differences",
    "book_tag": "WAHH",
    "topic": "XSS Classification & Context"
  },
  {
    "id": "q11",
    "query": "DOM based XSS taint flow sources sinks location hash document write",
    "book_tag": "WAHH",
    "topic": "DOM-based XSS Sinks & Sources"
  },
  {
    "id": "q12",
    "query": "cross site request forgery token validation bypass double submit cookie",
    "book_tag": "WAHH",
    "topic": "CSRF Defenses and Bypasses"
  },
  {
    "id": "q13",
    "query": "padding oracle attack cbc mode block cipher decryption without key",
    "book_tag": "WAHH",
    "topic": "Cryptographic Padding Oracle"
  },
  {
    "id": "q14",
    "query": "session fixation attack user authentication privilege escalation",
    "book_tag": "WAHH",
    "topic": "Session Management & Fixation"
  },
  {
    "id": "q15",
    "query": "bypassing path traversal restrictions using null byte injection url encoding",
    "book_tag": "WAHH",
    "topic": "Directory Traversal Evasion"
  }
]
""")
conceptual_queries = json.loads("""[
  {
    "id": "q01",
    "query": "escalating a compromised process to the highest administrative system level without providing credentials on windows",
    "book_tag": "Metasploit",
    "topic": "Privilege Escalation in Meterpreter"
  },
  {
    "id": "q02",
    "query": "mutating binary payload instructions into polymorphic variants to bypass antivirus inspection and avoid null bytes",
    "book_tag": "Metasploit",
    "topic": "Payload Encoding and Evasion"
  },
  {
    "id": "q03",
    "query": "creating a standalone executable that connects back over the network to establish an interactive remote control shell",
    "book_tag": "Metasploit",
    "topic": "Payload Generation"
  },
  {
    "id": "q04",
    "query": "identifying which database engine backend is running based on how the parser handles query remark symbols",
    "book_tag": "SQLi_Attacks",
    "topic": "SQL Dialect Fingerprinting"
  },
  {
    "id": "q05",
    "query": "inferring secret database contents one character at a time by making the server pause before sending the response",
    "book_tag": "SQLi_Attacks",
    "topic": "Time-based Blind SQLi"
  },
  {
    "id": "q06",
    "query": "saving malicious data into user profile records that trigger code evaluation only when reviewed by an administrator",
    "book_tag": "SQLi_Attacks",
    "topic": "Second-Order SQL Injection"
  },
  {
    "id": "q07",
    "query": "extracting sensitive database rows across firewalls by forcing the database server to resolve network domain names",
    "book_tag": "SQLi_Attacks",
    "topic": "Out-of-band Data Exfiltration"
  },
  {
    "id": "q08",
    "query": "evading perimeter traffic filters by splitting malicious database commands across inline multi-line remark blocks",
    "book_tag": "SQLi_Attacks",
    "topic": "WAF Evasion via Comment Obfuscation"
  },
  {
    "id": "q09",
    "query": "extracting information when input is placed directly into the sorting parameter where union operators are forbidden",
    "book_tag": "SQLi_Attacks",
    "topic": "Order By Clause Injection"
  },
  {
    "id": "q10",
    "query": "how victim interaction differs between malicious scripts echoed back immediately versus those permanently saved on the server",
    "book_tag": "WAHH",
    "topic": "XSS Classification & Context"
  },
  {
    "id": "q11",
    "query": "untrusted client side input flowing into dynamic browser execution methods without round tripping through the web server",
    "book_tag": "WAHH",
    "topic": "DOM-based XSS Sinks & Sources"
  },
  {
    "id": "q12",
    "query": "tricking an authenticated user's browser into submitting unauthorized state changing actions without their knowledge",
    "book_tag": "WAHH",
    "topic": "CSRF Defenses and Bypasses"
  },
  {
    "id": "q13",
    "query": "decrypting encrypted data blocks without knowing the secret key by analyzing error responses from validation routines",
    "book_tag": "WAHH",
    "topic": "Cryptographic Padding Oracle"
  },
  {
    "id": "q14",
    "query": "forcing a victim to adopt an attacker chosen identifier before logging in so the attacker retains access afterwards",
    "book_tag": "WAHH",
    "topic": "Session Management & Fixation"
  },
  {
    "id": "q15",
    "query": "accessing restricted files outside the intended web root folder by terminating the file path string prematurely",
    "book_tag": "WAHH",
    "topic": "Directory Traversal Evasion"
  }
]
""")

print(f"✓ Loaded {len(corpus):,} chunks across {len(set(c['book'] for c in corpus))} books")
print(f"✓ Loaded {len(exact_queries)} exact technical queries")
print(f"✓ Loaded {len(conceptual_queries)} conceptual queries")

In [ ]:
# 3. Verify Dual T4 GPUs
import torch
num_gpus = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {num_gpus}")
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.1f} GB")

if num_gpus < 2:
    print("\n⚠️ WARNING: Less than 2 GPUs detected! In Kaggle sidebar, set Accelerator to 'GPU T4 x2'.")

In [ ]:
# 4. Load pplx-embed-v2-context-9b-preview across Dual GPUs
import os, time
from transformers import AutoModel, AutoTokenizer

print("Loading perplexity-ai/pplx-embed-v2-context-9b-preview in unquantized FP16 across Dual GPUs...")
t0 = time.perf_counter()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

model = AutoModel.from_pretrained(
    "perplexity-ai/pplx-embed-v2-context-9b-preview",
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "9GB", 1: "13GB"},
    trust_remote_code=True,
)

# Explicitly attach tokenizer with trust_remote_code=True to bypass custom model property bug
tokenizer = AutoTokenizer.from_pretrained(
    "perplexity-ai/pplx-embed-v2-context-9b-preview",
    trust_remote_code=True,
    padding_side="right",
)
model.tokenizer = tokenizer

# Multi-GPU device alignment patch for custom _pool method
import torch.nn.functional as F, types
def _safe_pool(self, features: dict, task: str):
    dev0 = self.device
    input_ids = features["input_ids"].to(dev0)
    attention_mask = features["attention_mask"].to(dev0)
    hidden = self(input_ids=input_ids, attention_mask=attention_mask, use_cache=False, return_dict=True).last_hidden_state
    dev_out = hidden.device
    with torch.autocast(device_type=dev_out.type, enabled=False):
        hidden = hidden.float()
        if task == "query":
            mask = attention_mask.to(dev_out).unsqueeze(-1).float()
            pooled = ((hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)).unsqueeze(1)
        else:
            prefix = torch.cat((hidden.new_zeros(hidden.shape[0], 1, hidden.shape[-1]), hidden.cumsum(dim=1)), dim=1)
            spans = features["chunk_token_spans"].to(dev_out)
            starts, ends = spans.unbind(dim=-1)
            rows = torch.arange(hidden.shape[0], device=dev_out).unsqueeze(1)
            pooled = (prefix[rows, ends] - prefix[rows, starts]) / (ends - starts).clamp_min(1).unsqueeze(-1)
        proj_w = self.contextual_projection.weight.to(dev_out).float()
        return F.linear(pooled, proj_w)

model._pool = types.MethodType(_safe_pool, model)
print("✓ Multi-GPU device alignment patch applied to model._pool")

load_time = time.perf_counter() - t0
print(f"✓ Model & Tokenizer loaded in {load_time:.1f}s")
if hasattr(model, 'hf_device_map'):
    print(f"Device map layer distribution:")
    devices = set(model.hf_device_map.values())
    for d in sorted(list(devices)):
        layers = [k for k, v in model.hf_device_map.items() if v == d]
        print(f"  Device {d}: {len(layers)} modules")

In [ ]:
# 5. Contextual Document Chunking & Encoding
import numpy as np, gc, torch

# Free any lingering fragmentation from previous runs
gc.collect()
torch.cuda.empty_cache()

# Group contiguous chunks by section (capped at 2 chunks per document to keep token sequence short and prevent OOM)
doc_chunks = []
current_key = None
current_doc = []

for idx, c in enumerate(corpus):
    key = (c['book_tag'], c['section'])
    if key != current_key or len(current_doc) >= 2:
        if current_doc:
            doc_chunks.append(current_doc)
        current_doc = []
        current_key = key
    current_doc.append(f"title: {c['title']} | text: {c['text']}")

if current_doc:
    doc_chunks.append(current_doc)

total_chunks = sum(len(d) for d in doc_chunks)
print(f"Grouped {total_chunks} chunks into {len(doc_chunks)} lightweight contextual clusters (max 2 chunks each).")

print("Encoding document clusters with shared attention (batch_size=1)...")
t0 = time.perf_counter()
# Batch size 1 eliminates activation memory spikes across Dual T4 cards
doc_embeddings = model.encode(doc_chunks, batch_size=1, normalize_embeddings=True, show_progress_bar=True)
context_doc_time = time.perf_counter() - t0

corpus_embeddings = np.vstack(doc_embeddings)
print(f"✓ Contextual encoding finished in {context_doc_time:.1f}s. Matrix shape: {corpus_embeddings.shape}")

In [ ]:
# 6. Evaluate Exact & Conceptual Queries
# Encode exact queries
print("Encoding exact technical queries...")
eq_input = [[q['query']] for q in exact_queries]
eq_vecs = np.vstack(model.encode_queries(eq_input, normalize_embeddings=True))
eq_sims = eq_vecs @ corpus_embeddings.T

exact_results = {}
for i, q in enumerate(exact_queries):
    row = eq_sims[i]
    top = row.argsort()[::-1][:10]
    exact_results[q['id']] = [(top[j], float(row[top[j]])) for j in range(10)]

# Encode conceptual queries
print("Encoding conceptual zero-keyword queries...")
cq_input = [[q['query']] for q in conceptual_queries]
cq_vecs = np.vstack(model.encode_queries(cq_input, normalize_embeddings=True))
cq_sims = cq_vecs @ corpus_embeddings.T

concept_results = {}
for i, q in enumerate(conceptual_queries):
    row = cq_sims[i]
    top = row.argsort()[::-1][:10]
    concept_results[q['id']] = [(top[j], float(row[top[j]])) for j in range(10)]

# Compute Accuracies
p9b_exact_acc = sum(1 for q in exact_queries if corpus[exact_results[q['id']][0][0]]['book_tag'] == q['book_tag']) / len(exact_queries)
p9b_concept_acc = sum(1 for q in conceptual_queries if corpus[concept_results[q['id']][0][0]]['book_tag'] == q['book_tag']) / len(conceptual_queries)

print("✓ Query evaluation complete!")

In [ ]:
# 7. Comprehensive 4-Way Benchmark Table
print("=" * 85)
print("COMPREHENSIVE BENCHMARK: EXACT vs CONCEPTUAL RETRIEVAL")
print("=" * 85)

# Reference baselines from previous tests:
# BM25: Exact 93.3%, Conceptual 60.0%
# Gemma 2 (270M): Exact 86.7%, Conceptual 86.7%
# Late-0.6b (340M): Exact 86.7%, Conceptual 80.0%

print(f"{'Model':<30} {'Exact Top-1':<15} {'Conceptual Top-1':<18} {'Delta':<10}")
print("-" * 85)
print(f"{'BM25':<30} {'93.3%':<15} {'60.0%':<18} {'-33.3%':<10}")
print(f"{'EmbeddingGemma 2 (270M)':<30} {'86.7%':<15} {'86.7%':<18} {'+0.0%':<10}")
print(f"{'Late Interaction (0.6B)':<30} {'86.7%':<15} {'80.0%':<18} {'-6.7%':<10}")
delta_str = f"{(p9b_concept_acc - p9b_exact_acc)*100:+5.1f}%"
print(f"{'Pplx Contextual (9B)':<30} {f'{p9b_exact_acc*100:.1f}%':<15} {f'{p9b_concept_acc*100:.1f}%':<18} {delta_str:<10}")
print("=" * 85)

print("\nSAMPLE SIDE-BY-SIDE: CONCEPTUAL QUERY 1")
q1 = conceptual_queries[0]
top1_idx, sc1 = concept_results[q1['id']][0]
hit1 = corpus[top1_idx]
print(f"Query: \"{q1['query']}\"")
print(f"Target: [{q1['book_tag']}] {q1['topic']}")
print(f"Pplx-9B Top-1: [{hit1['book_tag']} p.{hit1['pages'][0]}] {hit1['section']} (score: {sc1:.3f})")
print(f"Text: \"{hit1['text'][:120]}...\"")